[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/dirac_kahler/dirac_kahler.ipynb)

# Dirac–Kähler Whitney Fields

This example computes mixed-grade Dirac–Kähler derivatives, Whitney cochain reconstructions, three-way Hodge–Helmholtz flow decompositions, and boundary-coupled waves on triangulated planar domains. A Dirac–Kähler field holds a scalar potential, a 1-form, and a stream bivector together at each vertex so one derivative acts across all grades at once. In a wave of that field, the scalar grade spreads like sound while the bivector grade circulates like an eddy; the two travel independently across the interior, whereas at the rims of a perforated annulus an incoming pressure pulse sheds part of its energy into circulation. On the same perforated annulus, a flow splits into a potential part from sources to sinks, a local eddy part that closes within the domain, and a harmonic circulation winding around the hole.

The term ***simplex gradient*** here means the constant slope across a triangle that rises from zero on an opposite edge to one at its corner. Multiplying the corner simplex gradients by `Full` combines divergence in grade 0, potential slope plus stream slope in grade 1, and curl in grade 2 in one geometric product. Restricting that product to `Even` couples scalar and stream potentials into a single face 1-form while its adjugate returns divergence and circulation to the vertices, and meeting a stream slope with the triangle patch `Plane ^ patches` turns it into a velocity along level curves without a metric.

In differential-form and finite-element exterior-calculus notation, the Dirac–Kähler operator on inhomogeneous forms $\Phi = \phi_0 + \alpha_1 + \omega_2$ is $d + \delta$ with formal codifferential $\delta = -{\star d \star}$, squaring to the Hodge Laplacian $(d + \delta)^2 = d\delta + \delta d$, while staggered cochains reconstruct via separate Whitney maps $W_0 \phi + W_1 u + W_2 \omega$ on vertices, edges, and faces and the Hodge–Helmholtz split of a flux 1-form reads $\iota_{\mathbf{v}} \mathrm{vol} = {\star d\phi} + d\psi + h$ across $L^2\Omega^1(\Omega) = dH^1_0(\Omega) \oplus {\star d H^1_0(\Omega)} \oplus \mathcal{H}^1(\Omega)$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA2D as ga
from numga.extensor import Extensor
from numga.sparse import spdiag
from numga.topology import Topology
from examples.util import shapes
from examples.surfaces.dirac_kahler import render, scenarios
from examples.util.animation import save_figure, show_states

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                  # Scalar+
Plane = ga.gatype.vector()                                                   # Vector+
Bivector = ga.gatype.bivector()                                              # Bivector+
Even = ga.gatype.even()                                                      # Even+
Full = ga.gatype.full()                                                      # Full+
Point = ga.gatype.antivector()                                               # Antivector-

np.set_printoptions(precision=4, suppress=True)

## Mixed-grade derivatives and Whitney maps

Scalar potentials produce slopes that cross their level curves, whereas stream bivectors produce slopes that follow them. Both enter the grade-one part of the Dirac–Kähler derivative on a triangulated disc.

In [ ]:
n_disk_divisions = 9
disk_radius = 1.0

In [ ]:
def simplex_geometry(topology: Topology, vertices: Point) -> tuple[Extensor, Extensor, Scalar, Plane]:
    """Opposite triangle edges and signed patches determine the face areas and corner simplex gradients (`[3] Plane+[F]`)."""
    edges = topology.d[0] * vertices                                         # Point-[E]
    sides = topology.boundary[2, 1] * edges                                  # [3] Point-[F]
    patches = (sides[0] & sides[1]) / 2                                      # Antibivector-[F]
    areas = patches.norm()                                                   # Scalar+[F]
    # Dividing each opposite edge by twice the signed triangle patch gives the three constant simplex gradients.
    gradients = sides / (2 * patches)                                        # [3] Plane+[F]
    return sides, patches, areas, gradients


def dirac_derivatives(topology: Topology, gradients: Plane) -> tuple[Extensor, Extensor, Extensor]:
    """Summing the simplex gradients over each triangle's corners yields the scalar gradient, the mixed-grade Dirac–Kähler derivative, and its `Even` restriction."""
    corners = topology.incidence[2, 0]                                       # [3] Scalar+[F <- V]
    gradient = (spdiag(gradients) * corners).sum(axis=0) * Scalar            # (Plane+ <- Scalar+)[F <- V]
    # Multiplying the simplex gradients by a full multivector unites divergence in grade 0, gradient plus stream slope in grade 1, and curl in grade 2.
    dirac = (spdiag(gradients) * corners).sum(axis=0) * Full                 # (Full+ <- Full+)[F <- V]
    even_gradient = (spdiag(gradients) * corners).sum(axis=0) * Even         # (Plane+ <- Even+)[F <- V]
    return gradient, dirac, even_gradient


disk_topology, disk_coordinates = shapes.hexagonal_disk(disk_radius, n_disk_divisions)
disk_vertices = mv.vector(disk_coordinates).dual().field()                   # Point-[V]
_, disk_patches, _, disk_gradients = simplex_geometry(disk_topology, disk_vertices)
disk_gradient, dirac, disk_even_gradient = dirac_derivatives(disk_topology, disk_gradients)

x, y = mv.x & disk_vertices, mv.y & disk_vertices                            # Scalar+[V], Scalar+[V]
scalar_potential = 0.5 * (x.squared() - y.squared()) + 0.35 * x              # Scalar+[V]
vertex_1form = (0.6 * x - 0.8 * y) * mv.x + (0.8 * x + 0.6 * y) * mv.y       # Plane+[V]
stream_bivector = (x * y - 0.35 * y) * mv.xy                                 # Bivector+[V]
vertex_multivector = scalar_potential + vertex_1form + stream_bivector       # Full+[V]
scalar_gradient = disk_gradient(scalar_potential)                            # Plane+[F]
face_dirac = dirac(vertex_multivector)                                       # Full+[F]

A staggered field instead stores the flow by its integrals along the edges and their circulation around each face.

In [ ]:
def whitney_maps(
    topology: Topology, patches: Extensor, gradients: Plane,
) -> tuple[Extensor, Extensor, Extensor, Extensor]:
    """Corner means and simplex-gradient differences embed 0-, 1-, and 2-cochains into face fields alongside the exterior coboundary."""
    exterior_curl = topology.d[1] * Scalar                                   # (Scalar+ <- Scalar+)[F <- E]
    corner_mean = topology.incidence[2, 0].mean(axis=0)                      # Scalar+[F <- V]
    whitney0 = corner_mean * Scalar                                          # (Scalar+ <- Scalar+)[F <- V]
    # At each triangle centroid the three corner weights equal 1/3, so each edge 1-form is one third of its endpoints' simplex-gradient difference.
    tails, heads = topology.corners[2, 1]
    centroid_forms = (gradients[heads] - gradients[tails]) / 3               # [3] Plane+[F]
    whitney1 = (spdiag(centroid_forms) * topology.oriented[2, 1]).sum(axis=0) * Scalar  # (Plane+ <- Scalar+)[F <- E]
    whitney2 = mv.xy / (mv.xy & patches) * Scalar                            # (Bivector+ <- Scalar+)[F]
    return exterior_curl, whitney0, whitney1, whitney2


exterior_curl, whitney0, whitney1, whitney2 = whitney_maps(disk_topology, disk_patches, disk_gradients)
edge_midpoints = disk_topology.incidence[1, 0].mean(axis=0) * disk_vertices  # Point-[E]
midpoint_x, midpoint_y = mv.x & edge_midpoints, mv.y & edge_midpoints        # Scalar+[E], Scalar+[E]
midpoint_1form = (0.6 * midpoint_x - 0.8 * midpoint_y) * mv.x + (0.8 * midpoint_x + 0.6 * midpoint_y) * mv.y  # Plane+[E]
edge_cochain = midpoint_1form & (disk_topology.d[0] * disk_vertices)         # Scalar+[E]
face_cochain = exterior_curl(edge_cochain)                                   # Scalar+[F]
whitney_multivector = whitney0(scalar_potential) + whitney1(edge_cochain) + whitney2(face_cochain)  # Full+[F]

save_figure(
    render.grades_figure(
        disk_topology, disk_vertices, vertex_multivector,
        scalar_gradient, face_dirac, whitney_multivector,
    ),
    "dirac_kahler_grades",
);

The scalar gradient crosses the potential contours (upper left, red) while the stream slope circulates along its level curves (upper right, blue), and their two saddle patterns sum to the uniform grade-one part of the Dirac–Kähler derivative (lower left). The lower-right panel shows the Whitney reconstruction from staggered edge integrals for comparison; downstream-fading streaks share one speed scale across all four panels.

In finite-element exterior-calculus notation, the staggered reconstruction reads $W_0\phi + W_1 u + W_2\omega$ with degrees of freedom assigned separately to vertices, edges, and faces.

In [ ]:
print("Mean divergence (grade 0 of dirac):", face_dirac.select[0].batch().mean().to_array())
print("Mean curl density (grade 2 of dirac):", (mv.xy.dual() & face_dirac.select[2].batch().mean()).to_array())

## Hodge–Helmholtz decomposition around a hole

Local eddies and circulation around the hole look alike in a small neighbourhood, yet an eddy closes inside the domain while a harmonic loop winds around the missing region. A scalar potential and a stream function pinned to zero on both rims capture the local flow, whereas net circulation requires distinct stream values on the inner and outer rims.

Each triangle carries an intrinsic velocity `Point`. Meeting a stream slope with the triangle patch `Plane ^ patches` turns it along level curves without a metric, while only the potential slope passes through the metric medium to drive its source-to-sink velocity.

In [ ]:
inner_radius = 0.32
outer_radius = 1.0
n_annulus_rings = 15
circulation_strength = 0.65

In [ ]:
def medium_operators(
    topology: Topology, patches: Extensor, areas: Scalar, gradient: Extensor, even_gradient: Extensor,
) -> tuple[Extensor, Extensor, Extensor, Extensor, Extensor, Extensor, Extensor]:
    """The isotropic metric medium and metric-free patch wedge build the potential, stream, and `Even` velocity maps and their Laplacians."""
    # An isotropic medium carries each face's slope through the metric to the area-weighted velocity it drives; its inverse reads back the driving slope.
    isotropic_medium = spdiag(areas * Plane.dual())                          # (Point- <- Plane+)[F <- F]
    isotropic_resistance = Plane.dual().inverse() / areas                    # (Plane+ <- Point-)[F]
    potential_velocity = isotropic_medium(gradient)                          # (Point- <- Scalar+)[F <- V]
    # Meeting a stream slope with the triangle patch gives the area-weighted velocity along level curves without a metric.
    stream_velocity = (Plane ^ patches)(gradient)                            # (Point- <- Scalar+)[F <- V]
    even_velocity = isotropic_medium(even_gradient)                          # (Point- <- Even+)[F <- V]
    stream_laplacian = stream_velocity.adjugate()(isotropic_resistance(stream_velocity))  # (Antiscalar- <- Scalar+)[V <- V]
    even_laplacian = even_gradient.adjugate()(even_velocity)                 # (Even- <- Even+)[V <- V]
    return (
        isotropic_medium, isotropic_resistance, potential_velocity,
        stream_velocity, even_velocity, stream_laplacian, even_laplacian,
    )


topology, vertices, inner_ring, interior = scenarios.perforated_annulus(inner_radius, outer_radius, n_annulus_rings)
_, patches, areas, gradients = simplex_geometry(topology, vertices)
annulus_gradient, _, even_gradient = dirac_derivatives(topology, gradients)
(
    isotropic_medium, isotropic_resistance, potential_velocity,
    stream_velocity, even_velocity, stream_laplacian, even_laplacian,
) = medium_operators(topology, patches, areas, annulus_gradient, even_gradient)

In [ ]:
def harmonic_circulation(
    topology: Topology, stream_velocity: Extensor, stream_laplacian: Extensor,
    inner_ring: np.ndarray, interior: np.ndarray,
) -> tuple[Scalar, Point]:
    """Pinning the stream function to one on the inner rim and zero on the outer rim yields the harmonic circulation velocity (`Point-[F]`)."""
    pinned = topology.reading(inner_ring, 0)                                 # Scalar+[V_inner <- V]
    boundary_stream = pinned.adjugate() * mv.scalar(np.ones(len(inner_ring))[..., None]).field()  # Scalar+[V]
    free_scalar = topology.reading(interior, 0).adjugate() * Scalar          # (Scalar+ <- Scalar+)[V <- V_interior]
    boundary_load = -free_scalar.adjugate()(stream_laplacian(boundary_stream))  # Antiscalar-[V_interior]
    interior_laplacian = free_scalar.adjugate()(stream_laplacian(free_scalar))  # (Antiscalar- <- Scalar+)[V_interior <- V_interior]
    harmonic_stream = boundary_stream + free_scalar(interior_laplacian.solve(boundary_load))  # Scalar+[V]
    return harmonic_stream, stream_velocity(harmonic_stream)


squared_radii = vertices.norm_squared()                                      # Scalar+[V]
bubble = (squared_radii - inner_radius**2) * (outer_radius**2 - squared_radii)  # Scalar+[V]
abscissa, ordinate = mv.x & vertices, mv.y & vertices                        # Scalar+[V], Scalar+[V]
true_potential = bubble * (-2.2 * abscissa + 0.9 * ordinate)                 # Scalar+[V]
true_stream = bubble * (1.6 * abscissa + 1.9 * ordinate)                     # Scalar+[V]
_, true_harmonic = harmonic_circulation(topology, stream_velocity, stream_laplacian, inner_ring, interior)
total_flow = (
    potential_velocity(true_potential)
    + stream_velocity(true_stream)
    + circulation_strength * true_harmonic
)                                                                            # Point-[F]

Starting from the total flow alone, one interior solve on the even subalgebra recovers both local potentials at once and leaves the harmonic circulation as the remainder.

In [ ]:
def hodge_decomposition(
    topology: Topology, even_gradient: Extensor, even_velocity: Extensor, even_laplacian: Extensor,
    total_flow: Point, interior: np.ndarray,
) -> tuple[Scalar, Scalar, Point, Point, Point]:
    """One interior `Even` Poisson solve recovers the scalar potential and stream function that split `total_flow` into potential, stream, and harmonic flows (`Point-[F]`)."""
    free_even = topology.reading(interior, 0).adjugate() * Even              # (Even+ <- Even+)[V <- V_interior]
    # On every interior vertex star, the adjugate of the even gradient reads divergence in grade 0 and circulation in grade 2 together.
    readings = free_even.adjugate()(even_gradient.adjugate()(total_flow))    # Even-[V_interior]
    interior_laplacian = free_even.adjugate()(even_laplacian(free_even))     # (Even- <- Even+)[V_interior <- V_interior]
    even_potential = free_even(interior_laplacian.solve(readings))           # Even+[V]
    potential = even_potential.select[0]                                     # Scalar+[V]
    stream = -(mv.xy.dual() & even_potential.select[2])                      # Scalar+[V]
    potential_flow = even_velocity(potential)                                # Point-[F]
    stream_flow = even_velocity(even_potential.select[2])                    # Point-[F]
    harmonic_flow = total_flow - potential_flow - stream_flow                # Point-[F]
    return potential, stream, potential_flow, stream_flow, harmonic_flow


potential, stream, potential_flow, stream_flow, harmonic_flow = hodge_decomposition(
    topology, even_gradient, even_velocity, even_laplacian, total_flow, interior,
)

save_figure(
    render.hodge_figure(
        topology, vertices, total_flow, potential, potential_flow, stream_flow, harmonic_flow,
    ),
    "dirac_kahler_hodge",
);

Because both local potentials vanish on the inner and outer rims, the three flows are mutually orthogonal in energy. The upper row shows the total velocity alongside its source-to-sink potential part; below, the stream part closes into local eddies (blue) while the harmonic part winds around the hole (purple). Downstream-fading streaks share one velocity scale across all four panels.

In Hodge–de Rham notation, the boundary-value split reads $L^2\Omega^1(\Omega) = dH^1_0(\Omega) \oplus {\star d H^1_0(\Omega)} \oplus \mathcal{H}^1(\Omega)$.

In [ ]:
potential_flow_energy = (isotropic_resistance(potential_flow) & potential_flow).sites.sum()  # [] Scalar+
stream_flow_energy = (isotropic_resistance(stream_flow) & stream_flow).sites.sum()  # [] Scalar+
harmonic_flow_energy = (isotropic_resistance(harmonic_flow) & harmonic_flow).sites.sum()  # [] Scalar+
print("Component energies (potential, stream, harmonic):", stack((potential_flow_energy, stream_flow_energy, harmonic_flow_energy)).to_array())

## Boundary mode conversion in a wave

A pure scalar pulse starts at rest with zero stream bivector and zero face 1-form. While the two even grades decouple across interior vertex stars, the inner and outer rims couple them as the pressure front splits around the hole and sheds part of its energy into a grade-two response through the skew-adjoint pairing of gradient and divergence. Stepping and rendering the wave animation takes about six seconds.

In [ ]:
horizontal, vertical = mv.x.dual(), mv.y.dual()                              # [] Point-, [] Point-
pulse_centre = -0.58 * horizontal + 0.18 * vertical                          # [] Point-
pulse_width = 0.15
dt = 0.00533
n_steps = 300
stride = 5
frame_duration_ms = 40

In [ ]:
def wave_divergence(
    topology: Topology, areas: Scalar, even_gradient: Extensor, isotropic_medium: Extensor,
) -> tuple[Scalar, Extensor]:
    """Dual-cell vertex areas and the negative adjugate of `even_gradient` through `isotropic_medium` form the skew-adjoint `Even` divergence."""
    dual_areas = topology.incidence[2, 0].mean(axis=0).adjugate() * areas    # Scalar+[V]
    inverse_vertex_mass = (1 / dual_areas) * Even.dual().inverse()           # (Even+ <- Even-)[V]
    # Minus the even gradient's adjugate through the medium makes divergence and gradient skew-adjoint partners in wave energy.
    even_divergence = -inverse_vertex_mass(even_gradient.adjugate()(isotropic_medium))  # (Even+ <- Plane+)[V <- F]
    return dual_areas, even_divergence


def wave_states(
    even_gradient: Extensor, even_divergence: Extensor, even_state: Even, face_field: Plane,
    dt: float, n_steps: int, stride: int,
) -> Iterator[tuple[Even, Plane]]:
    """Yields every `stride`-th fourth-order Runge–Kutta state (`Even+[V]`, `Plane+[F]`) of the coupled Dirac–Kähler wave."""
    yield even_state, face_field
    for step in range(1, n_steps + 1):
        first_even, first_field = even_divergence(face_field), even_gradient(even_state)  # Even+[V], Plane+[F]
        second_even = even_divergence(face_field + 0.5 * dt * first_field)   # Even+[V]
        second_field = even_gradient(even_state + 0.5 * dt * first_even)     # Plane+[F]
        third_even = even_divergence(face_field + 0.5 * dt * second_field)   # Even+[V]
        third_field = even_gradient(even_state + 0.5 * dt * second_even)     # Plane+[F]
        fourth_even = even_divergence(face_field + dt * third_field)         # Even+[V]
        fourth_field = even_gradient(even_state + dt * third_even)           # Plane+[F]
        even_state = even_state + (dt / 6) * (first_even + 2 * second_even + 2 * third_even + fourth_even)  # Even+[V]
        face_field = face_field + (dt / 6) * (first_field + 2 * second_field + 2 * third_field + fourth_field)  # Plane+[F]
        if step % stride == 0:
            yield even_state, face_field

In [ ]:
dual_areas, even_divergence = wave_divergence(topology, areas, even_gradient, isotropic_medium)
envelope = (-(vertices - pulse_centre).norm_squared() / (2 * pulse_width**2)).exp()  # Scalar+[V]
initial_even = envelope.cast(Even)                                           # Even+[V]
initial_face_field = even_gradient(initial_even * 0.0)                       # Plane+[F]

recorded_waves = show_states(
    wave_states(even_gradient, even_divergence, initial_even, initial_face_field, dt, n_steps, stride),
    render.waves_animation,
    topology,
    vertices,
    name="dirac_kahler_waves",
    duration_ms=frame_duration_ms,
    n_frames=n_steps // stride + 1,
);

The outer reliefs lift the signed scalar (left, red) and stream-bivector (right, blue) amplitudes on one shared height scale relative to the undisturbed rim, while the middle relief blends red scalar, green face 1-form, and blue bivector amplitudes. Relief elevation encodes field amplitude over the fixed triangulated annulus.

In [ ]:
# checks
corner_1form = disk_topology.incidence[2, 0] * vertex_1form                  # [3] Plane+[F]
corner_stream = disk_topology.incidence[2, 0] * stream_bivector              # [3] Bivector+[F]
div_1form = (disk_gradients | corner_1form).sum(axis=0)                      # Scalar+[F]
grad_plus_rot = scalar_gradient + (disk_gradients | corner_stream).sum(axis=0)  # Plane+[F]
curl_1form = (disk_gradients ^ corner_1form).sum(axis=0)                     # Bivector+[F]
assert ((face_dirac.select[0] - div_1form).abs() < 1e-12).all()
assert ((face_dirac.select[1] - grad_plus_rot).scalar_norm_squared() < 1e-24).all()
assert ((face_dirac.select[1] - disk_even_gradient(scalar_potential + stream_bivector)).scalar_norm_squared() < 1e-24).all()
assert ((face_dirac.select[2] - curl_1form).scalar_norm_squared() < 1e-24).all()
assert ((whitney_multivector.select[2] - face_dirac.select[2]).scalar_norm_squared() < 1e-24).all()

assert ((potential - true_potential).abs() < 1e-11).all()
assert ((stream - true_stream).abs() < 1e-11).all()
assert ((harmonic_flow - circulation_strength * true_harmonic).scalar_norm_squared() < 1e-22).all()
assert (isotropic_resistance(potential_flow) & stream_flow).sites.sum().abs() < 1e-11
assert (isotropic_resistance(potential_flow) & harmonic_flow).sites.sum().abs() < 1e-11
assert (isotropic_resistance(stream_flow) & harmonic_flow).sites.sum().abs() < 1e-11
total_flow_energy = (isotropic_resistance(total_flow) & total_flow).sites.sum()  # [] Scalar+
assert (potential_flow_energy + stream_flow_energy + harmonic_flow_energy - total_flow_energy).abs() < 1e-11

energies = stack([
    ((dual_areas * even_state.scalar_norm_squared()).sites.sum()
     + (areas * face_field.scalar_norm_squared()).sites.sum()) / 2
    for even_state, face_field in recorded_waves
])                                                                           # [...] Scalar+
final_vortical_energy = (dual_areas * recorded_waves[-1][0].select[2].scalar_norm_squared()).sites.sum() / 2  # [] Scalar+
assert ((energies / energies[0] - 1.0).abs() < 1e-4).all()
assert final_vortical_energy > 1e-4